# Session 12: Connecting Python to MongoDB with PyMongo

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-12_repository.py`

In [2]:
# Session 12 · A small data-access layer with PyMongo
from datetime import datetime
from pymongo import ASCENDING, ReturnDocument
from pymongo.errors import DuplicateKeyError, PyMongoError
from store_db import get_db

class ProductRepo:
    """All product database access in one place: the rest of the app never writes queries."""
    def __init__(self, db):
        self.col = db.products
        self.col.create_index([("category", ASCENDING), ("price", ASCENDING)])

    def add(self, doc):
        try:
            return self.col.insert_one(doc).inserted_id
        except DuplicateKeyError:
            raise ValueError(f"product {doc['_id']} already exists") from None

    def get(self, pid):
        return self.col.find_one({"_id": pid}, {"_id": 1, "name": 1, "price": 1, "stock": 1})

    def search(self, category=None, max_price=None, in_stock=True, page=1, size=2):
        q = {}
        if category: q["category"] = category
        if max_price is not None: q["price"] = {"$lte": max_price}
        if in_stock: q["stock"] = {"$gt": 0}
        cur = self.col.find(q, {"name": 1, "price": 1}).sort("price", 1).skip((page - 1) * size).limit(size)
        return list(cur)

    def sell(self, pid, qty):
        doc = self.col.find_one_and_update({"_id": pid, "stock": {"$gte": qty}},
                                           {"$inc": {"stock": -qty}, "$set": {"lastSold": datetime(2025, 6, 30)}},
                                           projection={"name": 1, "stock": 1},
                                           return_document=ReturnDocument.AFTER)
        if not doc:
            raise ValueError(f"cannot sell {qty} of product {pid}: not enough stock")
        return doc

repo = ProductRepo(get_db())
print("add:", repo.add({"_id": 114, "name": "Laptop Stand", "category": "Accessories", "price": 3500, "stock": 10}))
try:
    repo.add({"_id": 114, "name": "Duplicate"})
except ValueError as e:
    print("add again:", e)
print("get:", repo.get(114))
print("search Accessories <= 5000, page 1:", [p["name"] for p in repo.search("Accessories", 5000)])
print("search Accessories <= 5000, page 2:", [p["name"] for p in repo.search("Accessories", 5000, page=2)])
print("sell 3:", repo.sell(114, 3))
try:
    repo.sell(104, 1)
except ValueError as e:
    print("sell out-of-stock:", e)
except PyMongoError as e:                       # network/server problems land here on a real server
    print("database error:", e)

add: 114
add again: product 114 already exists
get: {'name': 'Laptop Stand', 'price': 3500, 'stock': 10, '_id': 114}
search Accessories <= 5000, page 1: ['Wireless Mouse', 'Laptop Stand']
search Accessories <= 5000, page 2: ['Webcam HD']
sell 3: {'name': 'Laptop Stand', 'stock': 7, '_id': 114}
sell out-of-stock: cannot sell 1 of product 104: not enough stock


## Example: `session-12_to_pandas.py`

In [3]:
# Session 12 · From MongoDB to pandas for analysis
import pandas as pd
from store_db import get_db

db = get_db()
pd.set_option("display.width", 120)

# 1) Let MongoDB do the heavy lifting, then hand pandas a small, tidy result
pipeline = [
    {"$match": {"status": {"$ne": "Cancelled"}}},
    {"$unwind": "$items"},
    {"$lookup": {"from": "products", "localField": "items.productId", "foreignField": "_id", "as": "p"}},
    {"$unwind": "$p"},
    {"$lookup": {"from": "customers", "localField": "customerId", "foreignField": "_id", "as": "c"}},
    {"$unwind": "$c"},
    {"$project": {"_id": 0, "orderId": "$_id", "date": "$orderDate", "city": "$c.city",
                  "category": "$p.category", "qty": "$items.qty",
                  "revenue": {"$multiply": ["$items.qty", "$items.price"]}}},
]
df = pd.DataFrame(list(db.orders.aggregate(pipeline)))
print("Order lines:", df.shape)
print(df.head().to_string(index=False))

print("\nRevenue by city and category (pivot table):")
print(df.pivot_table(index="city", columns="category", values="revenue", aggfunc="sum", fill_value=0).to_string())

print("\nMonthly revenue:")
print(df.groupby(df["date"].dt.to_period("M"))["revenue"].sum().to_string())

# 2) Nested documents -> flat columns with json_normalize
prod = pd.json_normalize(list(db.products.find({}, {"tags": 0})))
print("\nProducts flattened:", list(prod.columns))
print(prod.groupby("specs.brand")["price"].agg(["count", "mean"]).sort_values("count", ascending=False).head(3).to_string())

# 3) Write analysis results back to MongoDB
summary = df.groupby("category", as_index=False)["revenue"].sum()
db.category_summary.insert_many(summary.to_dict("records"))
print("\nSaved", db.category_summary.count_documents({}), "summary documents to kgs.category_summary")

Order lines: (14, 6)
 orderId       date    city    category  qty  revenue
    5001 2025-03-02 Karachi Accessories    2     3000
    5001 2025-03-02 Karachi Accessories    1     4500
    5002 2025-03-05  Lahore    Displays    1    52000
    5003 2025-03-18 Karachi       Audio    1    18500
    5003 2025-03-18 Karachi       Audio    1     6500

Revenue by city and category (pivot table):
category  Accessories  Audio  Computers  Displays  Storage
city                                                      
Karachi         25500  25000     210000     52000    32000
Lahore          12600  13000          0     52000        0

Monthly revenue:
date
2025-03     84500
2025-04     41000
2025-05    237100
2025-06     59500
Freq: M

Products flattened: ['_id', 'name', 'category', 'price', 'stock', 'specs.brand', 'specs.warrantyMonths', 'specs.ramGB']
             count      mean
specs.brand                 
Logi             2    3000.0
ASUS             1  285000.0
Anker            1    4200.0

Save

## Example: `session-12_etl_batch.py`

In [4]:
# Session 12 · A small ETL job: extract CSV, clean, load in batches with upserts
import csv, io, random
from itertools import islice
from pymongo import UpdateOne
from store_db import get_db

db = get_db()
random.seed(12)

# Extract: pretend this CSV arrived from a point-of-sale system (with some dirty rows)
raw = io.StringIO()
w = csv.writer(raw); w.writerow(["sku", "name", "price", "stock", "city"])
for i in range(1, 2501):
    price = random.choice([str(random.randint(500, 90000)), "", "N/A"]) if i % 50 == 0 else str(random.randint(500, 90000))
    w.writerow([f"SKU{i:05d}", f"Item {i}", price, random.randint(0, 100), random.choice(["karachi", "Karachi ", "LAHORE", "Islamabad"])])
raw.seek(0)

def clean(rows):
    """Transform: fix types and casing, skip rows that cannot be repaired."""
    for r in rows:
        try:
            price = int(r["price"])
        except ValueError:
            clean.skipped += 1
            continue
        yield {"_id": r["sku"], "name": r["name"].strip(), "price": price,
               "stock": int(r["stock"]), "city": r["city"].strip().title()}
clean.skipped = 0

def batches(it, size):
    it = iter(it)
    while chunk := list(islice(it, size)):
        yield chunk

loaded = 0
for n, chunk in enumerate(batches(clean(csv.DictReader(raw)), 1000), 1):
    ops = [UpdateOne({"_id": d["_id"]}, {"$set": d}, upsert=True) for d in chunk]   # idempotent: safe to rerun
    res = db.pos_items.bulk_write(ops, ordered=False)
    loaded += res.upserted_count + res.modified_count
    print(f"batch {n}: {len(chunk)} rows -> upserted {res.upserted_count}, modified {res.modified_count}")

print(f"\nloaded {loaded}, skipped {clean.skipped} dirty rows")

# Rerun the first batch: upserts make the job idempotent (nothing is duplicated)
raw.seek(0)
first = next(batches(clean(csv.DictReader(raw)), 1000))
res = db.pos_items.bulk_write([UpdateOne({"_id": d["_id"]}, {"$set": d}, upsert=True) for d in first])
print(f"rerun batch 1: upserted {res.upserted_count}, modified {res.modified_count}, total docs {db.pos_items.count_documents({})}")
db.pos_items.create_index("city")
print("rows per city:", {r["_id"]: r["n"] for r in db.pos_items.aggregate(
    [{"$group": {"_id": "$city", "n": {"$sum": 1}}}, {"$sort": {"_id": 1}}])})

batch 1: 1000 rows -> upserted 1000, modified 0


batch 2: 1000 rows -> upserted 1000, modified 0


batch 3: 460 rows -> upserted 460, modified 0

loaded 2460, skipped 40 dirty rows


rerun batch 1: upserted 0, modified 0, total docs 2460
rows per city: {'Islamabad': 615, 'Karachi': 1224, 'Lahore': 621}
